In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


### READING DATA AND PREPROCESSING OF DATA

In [2]:
df_Hierachical_prompting = pd.read_csv("/home/nguenang/Master_thesis/experiment_setup/results_Hierarchical_prompting.csv")
df_flat_prompting = pd.read_csv("/home/nguenang/Master_thesis/experiment_setup/results_Flat_prompting.csv")

In [3]:
df_combined = pd.concat([df_flat_prompting, df_Hierachical_prompting])

In [4]:
df_combined["score_per"] = (df_combined["score"]-1)/3 * 100
df_combined["score_per"] = (df_combined["score"]-1)/3 * 100

In [6]:
# df_cat1 = pd.read_csv("/home/nguenang/Master_thesis/experiment_setup/result_category1.csv")
# df_cat1 = df_cat1[df_cat1["summarization_prompt"] != "evaluation_awareprompt"]
# df_cat1.shape
# # df_cat1.describe()
# # df_cat1
# df_cat1[df_cat1["llm_summarizer"] == "llama4"].shape

In [7]:
# df_cat4_H = pd.read_csv("/home/nguenang/Master_thesis/experiment_setup/result_category4_H.csv")
# df_cat4_H.shape
# df_cat4_H[df_cat4_H["llm_summarizer"] == "llama4"].shape

In [8]:
# df_all_task = pd.concat([df_cat1, df_cat4_H])
# # df_all_task[df_all_task["llm_summarizer"] == "llama4"].shape
# df_all_task.shape
# df_all_task.head

In [9]:
# df_all_task_filtered = df_all_task[(df_all_task["task"] == "CLASSIFICATION") | (df_all_task["task"] == "REGRESSION") ]
# df_all_task_filtered.shape

In [10]:
# df_cat2 = pd.read_csv("/home/nguenang/Master_thesis/experiment_setup/result_category2.csv")
# df_cat2.shape
# df_cat2 = df_cat2[df_cat2["summarization_prompt"] != "evaluation_awareprompt"]
# df_cat2[df_cat2["judging_llm"] == "deepseek-r1:14b"].shape

In [11]:
# df_cat2 = pd.read_csv("/home/nguenang/Master_thesis/experiment_setup/result_category2.csv")
# df_cat2 = df_cat2[df_cat2["summarization_prompt"] != "evaluation_awareprompt"]
# df_cat5_H = pd.read_csv("/home/nguenang/Master_thesis/experiment_setup/result_category5_H.csv")
# df_cat5_H.shape
# df_2task_3llm = pd.concat([df_cat2, df_cat5_H])
# df_2task_3llm.shape


In [13]:
# df_F = pd.read_csv("/home/nguenang/Master_thesis/experiment_setup/results_F.csv")
# df_F.shape
# df_F_H = pd.read_csv("/home/nguenang/Master_thesis/experiment_setup/results_F_H.csv")
# df_F_H.shape

In [5]:
# df_cat3 = pd.read_csv("/home/nguenang/Master_thesis/experiment_setup/result_category3_autoskln.csv")
# df_cat3 = df_cat3[df_cat3["summarization_prompt"] != "evaluation_awareprompt"]
# df_cat6_H = pd.read_csv("/home/nguenang/Master_thesis/experiment_setup/result_category6_H_autoskln.csv")
# df_cat3.shape
# df_autoskln = pd.concat([df_cat3, df_cat6_H])
# df_autoskln.shape

In [4]:
# df_combined = pd.concat([df_all_task_filtered, df_2task_3llm, df_autoskln])
# df_combined.shape

In [7]:
# df_F["score"] = df_F["score"].replace(5, 4)
# df_F_H["score"] = df_F_H["score"].replace(5,4)

In [8]:
# df_F["score_per"] = (df_F["score"]-1)/3 * 100
# df_F_H["score_per"] = (df_F_H["score"]-1)/3 * 100

In [16]:
# df_combined = pd.concat([df_F,df_all_task])
# df_combined.shape

In [5]:
df_combined["judging_llm"].unique

<bound method Series.unique of 0               llama4
1               llama4
2               llama4
3               llama4
4         gpt-4.1-mini
            ...       
471             llama4
472    deepseek-r1:14b
473    deepseek-r1:14b
474    deepseek-r1:14b
475    deepseek-r1:14b
Name: judging_llm, Length: 3334, dtype: object>

In [8]:
df_combined = df_combined[(df_combined["task"] != "Regression") & (df_combined["dataset"] != "185_baseball_dataset") & (df_combined["judging_llm"] != "deepseek-r1:14b")]

In [9]:
# ----------------------------
# Mean score per metric
# ----------------------------

# Compute mean score for each prompt × metric
metric_scores = (
    df_combined
   
    .groupby(["summarization_prompt", "metric"])["score_per"]
    .mean()
    .unstack()
)

print(metric_scores)

metric                 Accuracy     Clarity  Completeness   Relevance
summarization_prompt                                                 
Hierachical prompt    75.757576   95.959596     44.444444   68.686869
chain_of_thought      90.909091   98.989899     56.565657   97.979798
fewshot               90.909091   98.989899     44.444444   89.898990
fewshot+instruction   88.888889   95.959596     46.464646   80.808081
zeroshot              96.969697  100.000000     46.464646   78.431373
zeroshot_CoT          96.969697  100.000000     50.505051   94.949495
zeroshot_instruction  95.959596   98.989899     48.484848  100.000000


### RQ1: How does the choice of the prompt variant affected the quality of the summary 
Did the prompt variant affected the quality of the summary ?? 

In [19]:
tasks = df_combined['task'].unique()
datasets = df_combined['dataset'].unique()
llms = df_combined['llm_summarizer'].unique()
prompts = df_combined['summarization_prompt'].unique()
metrics = df_combined['metric'].unique()

In [ ]:
#------------------------------
#  the overall performance of prompt 
#-------------------------------------


for metric in metrics:

    subset = df_combined[
        (df_combined["metric"] == metric)
        # & (df_2task_3llm["task"] == task)
        # & (df_2task_3llm["dataset"] == dataset)
        # & (df_2task_3llm["summarization_prompt"] == prompt)
    ]

    if subset.empty:
        continue
    
    data = []
    labels = []

    for prompt in prompts:
        prompts_scores = subset[subset["summarization_prompt"] == prompt]["score_per"]

        if len(prompts_scores) > 0:
            data.append(prompts_scores)
            labels.append(prompt)

    if not data:
        continue

    plt.figure(figsize=(6,5))
    plt.boxplot(data)
    plt.xticks(range(1, len(labels)+1), labels, rotation=45)
    plt.ylabel("Score %")
    plt.title(f"Score Distribution per Prompt\nMetric: {metric}")
    plt.tight_layout()
    plt.show()


In [ ]:
#-------------------------------------------#
#  How the prompt performance varied by task#
#-------------------------------------------#

metrics_of_interest = [
    "Accuracy",
    "Completeness",
    "Clarity",
    "Relevance"
]

task_of_interest = df_combined["task"].unique()

filtered = df_combined[df_combined["metric"].isin(metrics_of_interest)]

interaction_scores = {}

for metric in metrics_of_interest:
    interaction_scores[metric] = (
        filtered[filtered["metric"] == metric]
        .groupby(["summarization_prompt", "task"])["score_per"]
        .mean()
        .unstack()
    )

# # Example
# print(interaction_scores["Accuracy"])

import numpy as np
import matplotlib.pyplot as plt

for metric, table in interaction_scores.items():

    prompts = table.index.tolist()
    tasks = table.columns.tolist()

    angles = np.linspace(0, 2 * np.pi, len(prompts), endpoint=False).tolist()
    angles += angles[:1]

    plt.figure(figsize=(8, 8))
    ax = plt.axes(polar=True)

    for task in tasks:
        values = table[task].tolist()
        values += values[:1]

        ax.plot(angles, values, marker='o', linewidth=2, label=task)
        ax.fill(angles, values, alpha=0.15)

    ax.set_thetagrids(np.degrees(angles[:-1]), prompts)
    ax.set_title(f"Summary quality wrt : Prompt × Task ({metric})")
    ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1))

    plt.show()

In [ ]:
#-------------------------------------------#
#   Summary quality wrt Prompt x Dataset    #
#-------------------------------------------#

import numpy as np
import matplotlib.pyplot as plt

metrics_of_interest = [
    "Accuracy",
    "Completeness",
    "Clarity",
    "Relevance"
]

datasets_of_interest = df_combined["dataset"].unique()

interaction_scores = {}

# -----------------------------
# Build the tables
# -----------------------------
for dataset in datasets_of_interest:
    
    dataset_df = df_combined[
        (df_combined["dataset"] == dataset) &
        (df_combined["metric"].isin(metrics_of_interest))
    ]
    
    table = (
        dataset_df
        .groupby(["summarization_prompt", "metric"])["score_per"]
        .mean()
        .unstack()
    )
    
    interaction_scores[dataset] = table


# -----------------------------
# Create 4x4 radar grid
# -----------------------------

n_datasets = len(interaction_scores)
n_cols = 4
n_rows = int(np.ceil(n_datasets / n_cols))

fig, axes = plt.subplots(
    n_rows, 
    n_cols, 
    subplot_kw=dict(polar=True), 
    figsize=(20, 20)
)

axes = axes.flatten()

for ax, (dataset, table) in zip(axes, interaction_scores.items()):

    prompts = table.index.tolist()
    metrics = table.columns.tolist()

    angles = np.linspace(0, 2 * np.pi, len(metrics), endpoint=False).tolist()
    angles += angles[:1]

    for prompt in prompts:
        values = table.loc[prompt].tolist()
        values += values[:1]

        ax.plot(angles, values, linewidth=1)
        ax.fill(angles, values, alpha=0.1)

    ax.set_thetagrids(np.degrees(angles[:-1]), metrics)
    ax.set_title(dataset, size=10)


# Hide unused subplots (if <16 datasets)
for i in range(len(interaction_scores), len(axes)):
    fig.delaxes(axes[i])

plt.tight_layout()
plt.show()



In [26]:
metrics_of_interest = [
    "Accuracy",
    "Completeness",
    "Clarity",
    "Relevance"
]

datasets_of_interest = df_combined["dataset"].unique()
prompts_of_interest = df_combined["summarization_prompt"].unique()

dataset_tables = {}

for dataset in datasets_of_interest:
    
    dataset_df = df_combined[
        (df_combined["dataset"] == dataset) &
        (df_combined["metric"].isin(metrics_of_interest))
    ]
    
    table = (
        dataset_df
        .groupby(["metric", "summarization_prompt"])["score_per"]
        .mean()
        .unstack()
    )
    
    dataset_tables[dataset] = table


In [ ]:
n_datasets = len(dataset_tables)
n_cols = 4
n_rows = int(np.ceil(n_datasets / n_cols))

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    subplot_kw=dict(polar=True),
    figsize=(22, 18)
)

axes = axes.flatten()

for ax, (dataset, table) in zip(axes, dataset_tables.items()):

    prompts = table.columns.tolist()
    metrics = table.index.tolist()

    angles = np.linspace(0, 2 * np.pi, len(prompts), endpoint=False).tolist()
    angles += angles[:1]

    for metric in metrics:
        values = table.loc[metric].tolist()
        values += values[:1]

        ax.plot(angles, values, linewidth=2, label=metric)
        ax.fill(angles, values, alpha=0.10)

    ax.set_thetagrids(np.degrees(angles[:-1]), prompts)
    ax.set_title(dataset, fontsize=10)
    ax.set_ylim(0, 100)

    ax.legend(loc="upper right", bbox_to_anchor=(1.25, 1.1), fontsize=7)



# Hide unused axes
for i in range(len(dataset_tables), len(axes)):
    fig.delaxes(axes[i])

plt.tight_layout()
plt.show()


In [ ]:

import numpy as np
import matplotlib.pyplot as plt

metrics_of_interest = [
    "Accuracy",
    "Completeness",
    "Clarity",
    "Relevance"
]

datasets_of_interest = df_combined["dataset"].unique()

dataset_tables = {}

# ---------------------------------
# Build tables (metric x prompt)
# ---------------------------------
for dataset in datasets_of_interest:
    
    dataset_df = df_combined[
        (df_combined["dataset"] == dataset) &
        (df_combined["metric"].isin(metrics_of_interest))
    ]
    print(dataset_df)
    table = (
        dataset_df
        .groupby(["metric", "summarization_prompt"])["score_per"]
        .mean()
        .unstack()
    )
    
    dataset_tables[dataset] = table

print(dataset_tables[dataset])



# ---------------------------------
# Radar Grid (4 per row)
# ---------------------------------

n_datasets = len(dataset_tables)
n_cols = 4
n_rows = int(np.ceil(n_datasets / n_cols))

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    subplot_kw=dict(polar=True),
    figsize=(22, 18)
)

axes = axes.flatten()

legend_handles = None

for i, (ax, (dataset, table)) in enumerate(zip(axes, dataset_tables.items())):

    prompts = table.columns.tolist()
    metrics = table.index.tolist()

    angles = np.linspace(0, 2 * np.pi, len(prompts), endpoint=False).tolist()
    angles += angles[:1]

    for metric in metrics:
        values = table.loc[metric].tolist()
        values += values[:1]

        ax.plot(angles, values, linewidth=2, label=metric)
        ax.fill(angles, values, alpha=0.08)

    ax.set_thetagrids(np.degrees(angles[:-1]), prompts)
    ax.set_title(dataset, fontsize=10)
    ax.set_ylim(0, 100)

    # Collect legend handles only once (after full plotting)
    if i == 0:
        legend_handles = ax.get_legend_handles_labels()





# Hide unused axes
for i in range(len(dataset_tables), len(axes)):
    fig.delaxes(axes[i])


# ---------------------------------
# Global Legend (only once)
# ---------------------------------

handles, labels = legend_handles

fig.legend(
    handles,
    labels,
    loc="lower center",
    ncol=4,
    fontsize=11,
    bbox_to_anchor=(0.5, -0.02)
)

plt.tight_layout()
plt.subplots_adjust(bottom=0.08)
plt.show()


In [ ]:
#-----------------------------------------------#
#------- Summary quality wrt prompt x llm------#
#----------------------------------------------#
metrics_of_interest = [
    "Accuracy",
    "Completeness",
    "Clarity",
    "Relevance"
]

filtered = df_combined[df_combined["metric"].isin(metrics_of_interest)]

interaction_scores = {}

for metric in metrics_of_interest:
    interaction_scores[metric] = (
        filtered[filtered["metric"] == metric]
        .groupby(["summarization_prompt", "llm_summarizer"])["score"]
        .mean()
        .unstack()
    )

# Example
print(interaction_scores["Accuracy"])

metrics_of_interest = [
    "Accuracy",
    "Completeness",
    "Clarity",
    "Relevance"
]

filtered = df_combined[df_combined["metric"].isin(metrics_of_interest)]

interaction_scores = {}

for metric in metrics_of_interest:
    interaction_scores[metric] = (
        filtered[filtered["metric"] == metric]
        .groupby(["summarization_prompt", "llm_summarizer"])["score"]
        .mean()
        .unstack()
    )

# Example
print(interaction_scores["Accuracy"])

import numpy as np
import matplotlib.pyplot as plt

for metric, table in interaction_scores.items():

    prompts = table.index.tolist()
    llms = table.columns.tolist()

    angles = np.linspace(0, 2 * np.pi, len(prompts), endpoint=False).tolist()
    angles += angles[:1]

    plt.figure(figsize=(8, 8))
    ax = plt.axes(polar=True)

    for llm in llms:
        values = table[llm].tolist()
        values += values[:1]

        ax.plot(angles, values, marker='o', linewidth=2, label=llm)
        ax.fill(angles, values, alpha=0.15)

    ax.set_thetagrids(np.degrees(angles[:-1]), prompts)
    ax.set_title(f"Interaction Effect: Prompt × LLM ({metric})")
    ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1))

    plt.show()



### R2: The effect of llm on the summarization quality

In [19]:
df_2task_3llm['score_per'] = (df_2task_3llm['score']-1)/3 * 100

In [ ]:
#-------------------------------------------------#
#------The Summary quality per LLM in overall-----#
#-------------------------------------------------#


for metric in metrics:

    subset = df_2task_3llm[
        (df_2task_3llm["metric"] == metric)
        # & (df_2task_3llm["task"] == task)
        # & (df_2task_3llm["dataset"] == dataset)
        # & (df_2task_3llm["summarization_prompt"] == prompt)
    ]

    if subset.empty:
        continue
    
    data = []
    labels = []

    for llm in llms:
        llm_scores = subset[subset["llm_summarizer"] == llm]["score_per"]

        if len(llm_scores) > 0:
            data.append(llm_scores)
            labels.append(llm)

    if not data:
        continue

    plt.figure(figsize=(6,5))
    plt.boxplot(data)
    plt.xticks(range(1, len(labels)+1), labels, rotation=45)
    plt.ylabel("Score %")
    plt.title(f"Score Distribution per LLM\nMetric: {metric}")
    plt.tight_layout()
    plt.show()

In [ ]:
#-----------------------------------------#
#-----Summary quality per llm  X task-----#
#-----------------------------------------#

import numpy as np
import matplotlib.pyplot as plt

for task in tasks:

    subset_task = df_2task_3llm[df_2task_3llm["task"] == task]

    if subset_task.empty:
        continue

    plt.figure(figsize=(6,6))
    ax = plt.subplot(111, polar=True)

    angles = np.linspace(0, 2*np.pi, len(metrics), endpoint=False)
    angles = np.concatenate([angles, [angles[0]]])

    for llm in llms:

        subset_llm = subset_task[subset_task["llm_summarizer"] == llm]

        if subset_llm.empty:
            continue

        # aggregate across prompts
        mean_scores = (
            subset_llm.groupby("metric")["score"]
            .mean()
            .reindex(metrics, fill_value=0)
        )

        values = mean_scores.values
        values = np.concatenate([values, [values[0]]])

        ax.plot(angles, values, label=llm, linewidth=2)
        ax.fill(angles, values, alpha=0.05)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(metrics)

    # VERY IMPORTANT (paper standard)
    # ax.set_ylim(0, 1)

    ax.set_title(f" Task: {task}")
    ax.legend(loc="upper right", bbox_to_anchor=(1.25, 1.1))

    plt.tight_layout()
    plt.show()





In [43]:
#----------------------------------------------------#
#----------The summary quality per LLM X dataset-----#
#----------------------------------------------------#

import numpy as np
import matplotlib.pyplot as plt

metrics_of_interest = [
    "Accuracy",
    "Completeness",
    "Clarity",
    "Relevance"
]

datasets_of_interest = df_2task_3llm["dataset"].unique()
llms_of_interest = df_2task_3llm["llm_summarizer"].unique()

dataset_tables = {}

for dataset in datasets_of_interest:
    
    dataset_df = df_combined[
        (df_combined["dataset"] == dataset) &
        (df_combined["metric"].isin(metrics_of_interest))
    ]
    
    table = (
        dataset_df
        .groupby(["llm_summarizer", "metric"])["score_per"]
        .mean()
        .unstack()
    )
    
    dataset_tables[dataset] = table


In [ ]:
n_datasets = len(dataset_tables)
n_cols = 4
n_rows = int(np.ceil(n_datasets / n_cols))

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    subplot_kw=dict(polar=True),
    figsize=(22, 18)
)

axes = axes.flatten()

legend_handles = None

for i, (ax, (dataset, table)) in enumerate(zip(axes, dataset_tables.items())):

    metrics = table.columns.tolist()
    llms = table.index.tolist()

    angles = np.linspace(0, 2 * np.pi, len(metrics), endpoint=False).tolist()
    angles += angles[:1]

    for llm in llms:
        values = table.loc[llm].tolist()
        values += values[:1]

        ax.plot(angles, values, linewidth=2, label=llm)
        ax.fill(angles, values, alpha=0.08)

    ax.set_thetagrids(np.degrees(angles[:-1]), metrics)
    ax.set_title(dataset, fontsize=10)
    ax.set_ylim(0, 100)

    # collect legend once
    if i == 0:
        legend_handles = ax.get_legend_handles_labels()


# Remove empty axes if any
for j in range(len(dataset_tables), len(axes)):
    fig.delaxes(axes[j])


# Global legend
handles, labels = legend_handles
fig.legend(
    handles,
    labels,
    loc="lower center",
    ncol=len(labels),
    fontsize=11,
    bbox_to_anchor=(0.5, -0.02)
)

plt.tight_layout()
plt.subplots_adjust(bottom=0.08)
plt.show()


### Measure the Robustness of the summarization prompt 

In [ ]:
df_alpha_automl = df_all_task[(df_all_task["task"] == "REGRESSION") | (df_all_task["task"] == "CLASSIFICATION")]
df_alpha_automl.shape
df_alpha_automl["autoML"] = "alpha_autoML"
# df_alpha_automl


In [ ]:
df_autoskln.shape
df_autoskln["autoML"] = "auto_skln"
df_robust_test = pd.concat([df_alpha_automl, df_autoskln])
df_robust_test.shape

In [ ]:
df_robust_test["score_per"] = (df_robust_test["score"]-1)/3 * 100
df_robust_test

In [ ]:
#------------------------------
#  the overall per autoML
#-------------------------------------

automls = df_robust_test["autoML"].unique()
for metric in metrics:

    subset = df_robust_test[
        (df_robust_test["metric"] == metric)
        # & (df_2task_3llm["task"] == task)
        # & (df_2task_3llm["dataset"] == dataset)
        # & (df_2task_3llm["summarization_prompt"] == prompt)
    ]

    if subset.empty:
        continue
    
    data = []
    labels = []

    for automl in automls:
        automl_scores = subset[subset["autoML"] == automl]["score_per"]
        # print(automl_scores)
        if len(automl_scores) > 0:
            data.append(automl_scores)
            labels.append(automl)
            
            # print(data)

    # if not data:
    #     continue

    plt.figure(figsize=(6,5))
    plt.boxplot(data)
    plt.xticks(range(1, len(labels)+1), labels, rotation=45)
    plt.ylabel("Score %")
    plt.title(f"Score Distribution per AutoML\nMetric: {metric}")
    plt.tight_layout()
    plt.show()

In [ ]:
#------------------------------
#  the overall performance of prompt 
#-------------------------------------

metrics_of_interest = [
    "Accuracy",
    "Completeness",
    "Clarity",
    "Relevance"
]

task_of_interest = df_robust_test["task"].unique()

filtered = df_robust_test[df_robust_test["metric"].isin(metrics_of_interest)]

interaction_scores = {}

for metric in metrics_of_interest:
    interaction_scores[metric] = (
        filtered[filtered["metric"] == metric]
        .groupby(["summarization_prompt", "autoML"])["score_per"]
        .mean()
        .unstack()
    )

# # Example
# print(interaction_scores["Accuracy"])

import numpy as np
import matplotlib.pyplot as plt

for metric, table in interaction_scores.items():

    prompts = table.index.tolist()
    tasks = table.columns.tolist()

    angles = np.linspace(0, 2 * np.pi, len(prompts), endpoint=False).tolist()
    angles += angles[:1]

    plt.figure(figsize=(8, 8))
    ax = plt.axes(polar=True)

    for task in tasks:
        values = table[task].tolist()
        values += values[:1]

        ax.plot(angles, values, marker='o', linewidth=2, label=task)
        ax.fill(angles, values, alpha=0.15)

    ax.set_thetagrids(np.degrees(angles[:-1]), prompts)
    ax.set_title(f"Summary quality wrt : Prompt × AutoMl ({metric})")
    ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1))

    plt.show()

#### Judgement evaluation

In [ ]:
#------------------------------#
#-----overall distribution of the evaluation per llm-----#
#------------------------------#


for metric in metrics:

    subset = df_2task_3llm[
        (df_2task_3llm["metric"] == metric)
        # & (df_2task_3llm["task"] == task)
        # & (df_2task_3llm["dataset"] == dataset)
        # & (df_2task_3llm["summarization_prompt"] == prompt)
    ]

    if subset.empty:
        continue

    
    # summary = (
    #     subset
    #     .groupby("judging_llm")["score_per"]
    #     .agg(["count", "mean", "median", "std", "min", "max"])
    #     .round(2)
    # )

    # print(f"\nMetric: {metric}")
    # print(summary)
    
    
    summary_all = (
    df_2task_3llm
    .groupby(["metric", "judging_llm"])["score_per"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .round(2)
)

print(summary_all)

    
    # table = subset[["judging_llm", "score_per"]].sort_values("judging_llm")

    # print(f"\nMetric: {metric}")
    # print(table)
    
    
    
    
    
    
    # data = []
    # labels = []

    # for llm in llms:
    #     llm_scores = subset[subset["judging_llm"] == llm]["score_per"]

    #     if len(llm_scores) > 0:
    #         data.append(llm_scores)
    #         labels.append(llm)


    # if not data:
    #     continue

    # plt.figure(figsize=(6,5))
    # plt.boxplot(data)
    # plt.xticks(range(1, len(labels)+1), labels, rotation=45)
    # plt.ylabel("Score %")
    # plt.title(f"Score Distribution per Judging LLM\nMetric: {metric}")
    # plt.tight_layout()
    # plt.show()

In [55]:
# df_2task_3llm["sample_id"] = df_2task_3llm.groupby(['task', 'dataset', 'llm_summarizer', 'summarization_prompt',
#        'judging_llm', 'judging_prompt', 'metric', 'score','label','score_per']
#      ).cumcount()


In [53]:
# for metric in metrics:

#     subset = df_2task_3llm[df_2task_3llm["metric"] == metric]

#     # Pivot so each row has scores from all judges
#     pivot_df = subset.pivot_table(
#         index="sample_id",   # or your unique identifier
#         columns="judging_llm",
#         values="score_per"
#     )

#     # Drop rows where any judge is missing
#     pivot_df = pivot_df.dropna()

#     print(f"\nMetric: {metric}")
#     print(pivot_df.head())


In [54]:
# for metric in metrics:

#     subset = df_2task_3llm[df_2task_3llm["metric"] == metric]

#     pivot_df = subset.pivot_table(
#         index="sample_id",
#         columns="judging_llm",
#         values="score_per"
#     ).dropna()

#     correlation_matrix = pivot_df.corr(method="pearson")

#     print(f"\nPearson Correlation — Metric: {metric}")
#     print(correlation_matrix.round(3))
